# Step 3 - filter processed trajectories:
Input: data from step 2 

Filter for when there are 21 days spent in a location as “used”
export
Output:  'processed_trajectories_21dfilter.csv'

This step reduces the number of drifters from 1583 to 996

In [1]:
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import os

from shapely.geometry import Point, Polygon as ShapelyPolygon
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# File Paths to data
repo_path = '/Users/zephyrsylvester/repos/connectivity-manuscript/data/'
output_path = '/Users/zephyrsylvester/repos/connectivity-manuscript/processed_data/'

# List of simulations and locations
locations = ['BS', 'GERL', 'GP', 'MB2']


In [3]:
file = output_path + 'processed_trajectories_wpresence_unfiltered.csv'
processed_df=pd.read_csv(file) 
print('raw number of larvae:', processed_df.larval_id.nunique())
processed_df

raw number of larvae: 16658


,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2,outside
0,0,0,2016-11-01,-66.912544,287.83374,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
1,0,1,2016-11-02,-66.830450,288.03165,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
2,0,2,2016-11-03,-66.757600,288.15454,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
3,0,3,2016-11-04,-66.711136,288.16013,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
4,0,4,2016-11-05,-66.718590,288.06876,0,0,00_16_1_0001,2016,h_null,BS,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2998435,199,175,2019-09-12,-60.327457,306.02032,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
2998436,199,176,2019-09-13,-60.238560,305.90730,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
2998437,199,177,2019-09-14,-60.116886,305.83090,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True
2998438,199,178,2019-09-15,-60.023293,305.82330,10,4,00_18_1_0200,2018,h_null,BS,False,False,False,False,True


# Filter to 21 days in locs

In [4]:
# Define the presence columns
presence_columns = ['BS', 'GERL', 'GP', 'MB2','outside']

# Group by hypothesis, start year, IDL_loc, and larval_id, and sum the presence counts
grouped_df = processed_df.groupby(['hypothesis', 'start_year', 'IDL_loc', 'larval_id'])[presence_columns].sum().reset_index()

print('number of larvae:', grouped_df.larval_id.nunique())

grouped_df

number of larvae: 16658


,hypothesis,start_year,IDL_loc,larval_id,BS,GERL,GP,MB2,outside
0,h_dvm,2016,BS,02_16_1_0001,28,0,0,0,152
1,h_dvm,2016,BS,02_16_1_0002,13,0,0,0,167
2,h_dvm,2016,BS,02_16_1_0003,47,52,0,0,81
3,h_dvm,2016,BS,02_16_1_0004,33,0,0,0,147
4,h_dvm,2016,BS,02_16_1_0005,102,0,0,0,78
...,...,...,...,...,...,...,...,...,...
16653,h_size,2018,MB2,03_18_4_1071,0,0,31,58,91
16654,h_size,2018,MB2,03_18_4_1072,0,0,81,87,12
16655,h_size,2018,MB2,03_18_4_1073,0,0,69,102,9
16656,h_size,2018,MB2,03_18_4_1074,0,0,39,82,59


In [5]:
# Initialize an empty DataFrame to collect the filtered results
filtered_results = pd.DataFrame()

# Filter for each presence column where the IDL_loc matches the column name and the count is greater than 21
for col in presence_columns:
    presence_columns = ['BS', 'GERL', 'GP', 'MB2']
    filtered = grouped_df[(grouped_df['IDL_loc'] == col) & (grouped_df[col] > 21)]
    filtered_results = pd.concat([filtered_results, filtered])

# Extract the unique larval_id's from the filtered results
filtered_larval_ids = filtered_results['larval_id'].unique()

# Create a subset of processed_df containing only the filtered larval_id's
df_21 = processed_df[processed_df['larval_id'].isin(filtered_larval_ids)]
print('number of larvae:', df_21.larval_id.nunique())
df_21

number of larvae: 11487


,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2,outside
360,2,0,2016-11-01,-63.602314,298.26752,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
361,2,1,2016-11-02,-63.569042,298.23145,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
362,2,2,2016-11-03,-63.522503,298.25543,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
363,2,3,2016-11-04,-63.483067,298.28790,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
364,2,4,2016-11-05,-63.438553,298.33795,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2998255,198,175,2019-09-12,-61.923122,299.13315,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True
2998256,198,176,2019-09-13,-61.860650,299.25064,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True
2998257,198,177,2019-09-14,-61.813430,299.41430,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True
2998258,198,178,2019-09-15,-61.756500,299.58655,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True


In [6]:
# Export
output_file = os.path.join(output_path, 'processed_trajectories_21dfilter.csv')
df_21.to_csv(output_file, index=False)